# AI-Powered Flood Risk Prediction
# Model Evaluation
## Ghale

Evaluates the trained Logistic Regression and Random Forest with F1 and MCC on the same test set, then saves the best model to the models folder so the backend or the dashboard can load it.

In [1]:
!pip install pandas numpy scikit-learn joblib -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\kingn\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Load data and rebuild the same test set

In [2]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, matthews_corrcoef
import joblib

HERE = Path.cwd()
REPO = HERE if (HERE / "data").exists() else HERE.parent
DATA = REPO / "data"
NOTEBOOKS = REPO / "notebooks"
MODELS = REPO / "models"

df = pd.read_csv(DATA / "murray_bridge_river_level_historical.csv", skiprows=4,
                 names=["datetime", "water_level_m", "conductivity", "water_temp_c"])
df["datetime"] = pd.to_datetime(df["datetime"], format="%H:%M:%S %d/%m/%Y", errors="coerce")
df = df.dropna(subset=["datetime", "water_level_m"]).sort_values("datetime").reset_index(drop=True)

df["level_lag1"] = df["water_level_m"].shift(1)
df["level_lag2"] = df["water_level_m"].shift(2)
df["level_roll7"] = df["water_level_m"].shift(1).rolling(7).mean()
df["level_change3"] = df["water_level_m"].shift(1) - df["water_level_m"].shift(4)
df = df.dropna(subset=["level_lag1", "level_lag2", "level_roll7", "level_change3"])

threshold = df["water_level_m"].quantile(0.80)
df["high_risk"] = (df["water_level_m"] >= threshold).astype(int)

features = ["level_lag1", "level_lag2", "level_roll7", "level_change3"]
X = df[features]
y = df["high_risk"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Test rows:", len(X_test))

Test rows: 1272


## 2. Load the trained models

In [3]:
model_lr = joblib.load(NOTEBOOKS / "logistic_regression_real.joblib")
model_rf = joblib.load(NOTEBOOKS / "Random_Forest.joblib")

## 3. Evaluate both with F1 and MCC

In [4]:
def evaluate(model):
    pred = model.predict(X_test)
    return f1_score(y_test, pred), matthews_corrcoef(y_test, pred)


f1_lr, mcc_lr = evaluate(model_lr)
f1_rf, mcc_rf = evaluate(model_rf)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "F1": [round(f1_lr, 3), round(f1_rf, 3)],
    "MCC": [round(mcc_lr, 3), round(mcc_rf, 3)],
})
print(results.to_string(index=False))

              Model    F1   MCC
Logistic Regression 0.778 0.726
      Random Forest 0.795 0.746


## 4. Save the best model

In [5]:
MODELS.mkdir(exist_ok=True)
if mcc_rf >= mcc_lr:
    best_name, best_model = "Random Forest", model_rf
else:
    best_name, best_model = "Logistic Regression", model_lr

joblib.dump(best_model, MODELS / "best_model.joblib")
print("Best model:", best_name)
print("Saved models/best_model.joblib")

Best model: Random Forest
Saved models/best_model.joblib
